# Dagalog Jupyter Kernel — Introduction

This notebook demonstrates the **Dagalog** kernel for interactive RDF data pipelines.
Each cell is a pipeline step; the in-memory `Datastore` persists across all cells in a session.

## Setup

```bash
# Build and install the kernel (run from repo root)
cargo build -p dagalog-kernel
./target/debug/dagalog-kernel install

# Launch (pins root_dir to the repo root so relative paths in %%rml/%%load work —
# Jupyter would otherwise default the kernel's working directory to notebooks/,
# since that's where this notebook file lives)
jupyter lab --ServerApp.root_dir=. notebooks/dagalog_intro.ipynb
```

Select **Dagalog (SPARQL + RDF)** as the kernel.

## Step 1 — Load inline Turtle

`%%turtle` parses the cell body as Turtle and adds the triples to the session datastore.

In [ ]:
%%turtle
@prefix ex:   <http://example.com/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .

ex:Alice a foaf:Person ; foaf:name "Alice" ; foaf:age 30 ; ex:worksFor ex:Acme .
ex:Bob   a foaf:Person ; foaf:name "Bob"   ; foaf:age 25 ; ex:worksFor ex:Acme .
ex:Acme  a ex:Company  ; foaf:name "Acme Corp" .

## Step 2 — Query with SPARQL SELECT

Cells without a `%%` prefix are treated as SPARQL. SELECT results render as an HTML table.

In [ ]:
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX ex:   <http://example.com/>

SELECT ?person ?name ?age WHERE {
    ?person a foaf:Person ;
            foaf:name ?name ;
            foaf:age  ?age .
}
ORDER BY ?name

## Step 3 — Apply an RML CSV mapping

`%%rml` loads an [RML mapping file](https://rml.io/) and runs it against
the declared CSV source, adding the resulting triples to the session datastore.

Paths are relative to the kernel's working directory, which is `--ServerApp.root_dir`
(the repo root, per the setup instructions above) — **not** the directory the
notebook file lives in.

In [ ]:
%%rml tests/testdata/rml_persons_mapping.ttl

## Step 4 — OWL-RL Reasoning

`%%reason` runs OWL-RL materialisation on the current datastore, inferring new triples.

First, load a small OWL ontology that defines `ex:Employee rdfs:subClassOf foaf:Person`
and adds a new individual `ex:Carol` as an `ex:Employee`.
After reasoning, Carol will be inferred to be a `foaf:Person` too.

In [ ]:
%%turtle
@prefix ex:   <http://example.com/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

# OWL-RL subclass axiom: every Employee is a Person
ex:Employee rdfs:subClassOf foaf:Person .

# Carol is only asserted as an Employee — after reasoning she will be a Person too
ex:Carol a ex:Employee ; foaf:name "Carol" ; ex:worksFor ex:Acme .

In [ ]:
%%reason

## Step 5 — Inspect the graph after loading

Query to see all people known to the datastore.

In [ ]:
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX ex:   <http://example.com/>

# Carol was only asserted as ex:Employee — OWL-RL infers she is also foaf:Person
SELECT ?person ?name WHERE {
    ?person a foaf:Person ;
            foaf:name ?name .
}
ORDER BY ?name

## Step 6 — Inline Datalog rules

`%%datalog` parses and materialises forward-chaining Datalog rules.

In [ ]:
%%datalog
[?x, <http://example.com/colleague>, ?y] :-
    [?x, <http://example.com/worksFor>, ?org],
    [?y, <http://example.com/worksFor>, ?org] .

In [ ]:
PREFIX ex: <http://example.com/>

SELECT ?person ?colleague WHERE {
    ?person ex:colleague ?colleague .
    FILTER (?person != ?colleague)
}
ORDER BY ?person ?colleague

## Step 7 — OTTR template expansion

`%%ottr` parses inline stOTTR, expands all template instances, and adds the resulting
triples to the session datastore. Define the template and call it in the same cell.

In [ ]:
%%ottr
@prefix ex:   <http://example.com/> .
@prefix ottr: <http://ns.ottr.xyz/0.4/> .
@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .

ex:Person [ ottr:IRI ?person, xsd:string ?name ] :: {
  ottr:Triple (?person, rdf:type,  foaf:Person),
  ottr:Triple (?person, foaf:name, ?name)
} .

ex:Person(<http://example.com/John>, "John") .
ex:Person(<http://example.com/Dave>,  "Dave") .


In [ ]:
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX ex:   <http://example.com/>

SELECT ?person ?name WHERE {
    ?person a foaf:Person ;
            foaf:name ?name .
}
ORDER BY ?name
